# Notebook 17 — Interval-Aware CYP-Gated Experts

**Question:** Can the final two-model ensemble be calibrated to experimental confidence intervals, and can a CYP-conditioned soft mixture of residual experts improve interval-aware prediction?

Five pre-specified alternatives: point-Huber affine calibration, interval-Huber affine calibration, plus point-, interval-, and hybrid-loss **CYP-conditioned neural residual experts**. An internal two-fold scaffold-split selection chooses one model **using only outer training data**. The loss uses a differentiable approximation to interval distance, preserves a small point-regression term, penalizes overly large corrections, and softly supervises gating on the high-potency group.

The reported ST-RAE is a **local, unbootstrapped** interval-aware ratio; do not confuse it with the organizers' bootstrap-aggregated MA-ST-RAE. Because the run starts from saved Notebook 11 inner OOF predictions, it is CPU-first and does not fine-tune CheMeleon weights.

## 0. Install packages
Run once in Colab. Python ≥3.10 recommended. A GPU is optional for the neural or contrastive experiment; the pretrained CheMeleon model is **not retrained**.

In [ ]:
%pip -q install "numpy>=1.26" "pandas>=2.0" "rdkit>=2023.09" "scikit-learn>=1.4" "scipy>=1.11" "matplotlib>=3.8" "torch>=2.1"

## 1. Configure and upload Notebook 11 results
Download `cyp_notebook_11_results.zip` from the **real five-fold Notebook 11 run**, upload it to Colab, and set its path. Use a Google Drive `PERSISTENT_ROOT` to make interrupted executions restartable. Run `FAST_MODE=True` first; use `False` to produce research-scale tables.

In [ ]:
from pathlib import Path
import os, random
SEED = 2026
FAST_MODE = True  # first run uses outer fold 0; change to False for all five
FOLDS_TO_RUN = [0] if FAST_MODE else [0,1,2,3,4]
FP_BITS = 256  # choose 512 for final full run if CPU budget permits
NB11_RESULTS_ZIP = '/content/cyp_notebook_11_results.zip' if Path('/content').exists() else '/mnt/data/cyp_notebook_11_results.zip'
PERSISTENT_ROOT = None  # e.g. '/content/drive/MyDrive/CYP/PotencyRegime/17'
EXPERIMENT_TAG = 'regime_v1'  # increment after code/method changes to avoid stale caches
OUTPUT = (Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')) / 'cyp_notebook_17_results'
OUTPUT.mkdir(parents=True,exist_ok=True)
assert Path(NB11_RESULTS_ZIP).is_file(), 'Upload the REAL full five-fold Notebook 11 results ZIP and edit NB11_RESULTS_ZIP'
print('Notebook 17', 'FAST' if FAST_MODE else 'FIVE-FOLD', 'cache directory:', OUTPUT)


## 2. Load and audit the saved first-stage OOF predictions
The following standalone utilities parse the archive, confirm the baseline is the fixed 50:50 CheMeleon/screening blend, reconstruct inner-training CI bounds from the corresponding training-compound records, and **assert no outer held-out molecule or scaffold appears in the fitting data**.

In [ ]:
"""Common data handling and leakage-safe utilities embedded into standalone CYP Colab notebooks.
Input is the complete five-fold Notebook 11 results archive. No outer-fold targets enter fitting.
"""
import io, json, zipfile, hashlib, warnings, random, os, sys, math, time
from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from scipy.optimize import minimize
ISOS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
BASE_NAME='CheMeleon_finetuned + ScreenStack_LGBM'
CHEM_NAME='CheMeleon_finetuned'
SCREEN_NAME='ScreenStack_LGBM'
DESCRIPTOR_FNS=[('MolWt',Descriptors.MolWt),('MolLogP',Descriptors.MolLogP),('TPSA',Descriptors.TPSA),('NumHDonors',Descriptors.NumHDonors),('NumHAcceptors',Descriptors.NumHAcceptors),('NumRotatableBonds',Descriptors.NumRotatableBonds),('RingCount',Descriptors.RingCount),('FractionCSP3',Descriptors.FractionCSP3),('HeavyAtomCount',Descriptors.HeavyAtomCount),('NumAromaticRings',Descriptors.NumAromaticRings),('NumAliphaticRings',Descriptors.NumAliphaticRings),('NumHeteroatoms',Descriptors.NumHeteroatoms)]

def zread(z,leaf):
    paths=[n for n in z.namelist() if n == leaf or n.endswith('/'+leaf)]
    if len(paths)!=1:raise ValueError(f'{leaf}: expected exactly one archive member, found {paths}')
    return z.read(paths[0])

def load_archive(path,allow_incomplete=False):
    with zipfile.ZipFile(path) as z:
        outer=pd.read_csv(io.BytesIO(zread(z,'nested_outer_predictions.csv')))
        inner=pd.read_csv(io.BytesIO(zread(z,'inner_oof_predictions_for_selection.csv')))
        manifest=pd.read_csv(io.BytesIO(zread(z,'fold_manifest.csv')))
        metadata=json.loads(zread(z,'run_metadata.json'))
    if not allow_incomplete:
        assert metadata.get('scientifically_valid_full_model_run') and metadata.get('complete_five_fold_oof'), 'Need full scientifically valid Notebook 11 output, not proxy/FAST'
    assert {'canonical_smiles','scaffold_group','fold'}<=set(manifest.columns)
    assert manifest.canonical_smiles.is_unique and manifest.scaffold_group.notna().all()
    assert manifest.groupby('scaffold_group').fold.nunique().max()==1
    o=outer[outer.subset.isin([BASE_NAME,CHEM_NAME,SCREEN_NAME])].copy()
    assert set(o.subset.unique())=={BASE_NAME,CHEM_NAME,SCREEN_NAME},'Missing component predictions in Notebook 11 archive'
    assert o[['canonical_smiles','isoform','fold','subset']].duplicated().sum()==0
    names=['y_true','ci_low','ci_high','scaffold_group']
    baseline=o[o.subset==BASE_NAME][['canonical_smiles','isoform','fold','y_true','y_pred','ci_low','ci_high','scaffold_group']].copy().rename(columns={'y_pred':'base'})
    c=o[o.subset==CHEM_NAME][['canonical_smiles','isoform','fold','y_pred']].rename(columns={'y_pred':'chemeleon'})
    s=o[o.subset==SCREEN_NAME][['canonical_smiles','isoform','fold','y_pred']].rename(columns={'y_pred':'screenstack'})
    outer_data=baseline.merge(c,on=['canonical_smiles','isoform','fold'],validate='one_to_one').merge(s,on=['canonical_smiles','isoform','fold'],validate='one_to_one')
    # For this *fixed* candidate baseline use explicitly 50:50 rather than Notebook11's selected pair label.
    assert np.allclose(outer_data.base,0.5*(outer_data.chemeleon+outer_data.screenstack),atol=1e-6)
    assert outer_data.isna().sum().get('y_true',0)==0
    assert set(outer_data.isoform)==set(ISOS)
    inner=inner.copy()
    assert not inner.duplicated(['fold','canonical_smiles','isoform']).any()
    inner['base']=0.5*(inner[CHEM_NAME]+inner[SCREEN_NAME])
    inner['chemeleon']=inner[CHEM_NAME]
    inner['screenstack']=inner[SCREEN_NAME]
    # The outer-prediction file contains CI for every compound *in its own outer fold*.
    # Using that label metadata for molecules INSIDE the current training partition is valid.
    reference=outer_data[['canonical_smiles','isoform','y_true','ci_low','ci_high']]
    inner=inner.merge(reference,on=['canonical_smiles','isoform'],suffixes=('','_outer'),validate='many_to_one')
    assert np.allclose(inner.y_true,inner.y_true_outer,atol=1e-6)
    inner=inner.drop(columns=['y_true_outer'])
    inner=inner.merge(manifest[['canonical_smiles','scaffold_group','fold']].rename(columns={'fold':'true_outer_fold'}),on='canonical_smiles',validate='many_to_one')
    assert (inner.fold!=inner.true_outer_fold).all(),'Inner meta-training contains an outer held-out compound'
    assert (outer_data.groupby('canonical_smiles').fold.nunique()==1).all()
    for fold in sorted(outer_data.fold.unique()):
        train=inner[inner.fold==fold]; held=outer_data[outer_data.fold==fold]
        assert not set(train.canonical_smiles)&set(held.canonical_smiles)
        assert not set(train.scaffold_group)&set(held.scaffold_group)
    assert np.isfinite(outer_data[['base','chemeleon','screenstack','y_true']].to_numpy(float)).all()
    assert np.isfinite(inner[['base','chemeleon','screenstack','y_true']].to_numpy(float)).all()
    return outer_data,inner,manifest,metadata

def fingerprint_features(smiles_list,bits=512):
    gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=bits)
    X=np.empty((len(smiles_list),bits+len(DESCRIPTOR_FNS)),dtype=np.float32)
    fps=[]
    for i,s in enumerate(smiles_list):
        m=Chem.MolFromSmiles(s)
        if m is None:raise ValueError(f'Invalid SMILES: {s!r}')
        fp=gen.GetFingerprint(m);fps.append(fp)
        arr=np.zeros(bits,dtype=np.int8);DataStructs.ConvertToNumpyArray(fp,arr)
        X[i,:bits]=arr
        for j,(_,fn) in enumerate(DESCRIPTOR_FNS):
            try:v=fn(m);X[i,bits+j]=float(v) if np.isfinite(v) else 0.
            except Exception:X[i,bits+j]=0.
    return X,fps

def feature_table(outer,inner,bits=512):
    all_smiles=sorted(set(outer.canonical_smiles)|set(inner.canonical_smiles))
    arr,fps=fingerprint_features(all_smiles,bits)
    index={s:k for k,s in enumerate(all_smiles)}
    return {'x':arr,'fps':fps,'smiles':all_smiles,'index':index,'bits':bits}

def make_x(frame,features,include_model=True):
    ids=np.fromiter((features['index'][s] for s in frame.canonical_smiles),dtype=int,count=len(frame))
    desc=features['x'][ids]
    if include_model:
        additional=np.stack([frame.base.to_numpy(float),frame.chemeleon.to_numpy(float),frame.screenstack.to_numpy(float),
                             (frame.chemeleon-frame.screenstack).to_numpy(float),
                             np.asarray([ISOS.index(k) for k in frame.isoform],float)/3],axis=1).astype(np.float32)
        return np.column_stack([desc,additional])
    return desc

def metric_stats(frame,pred,original=None):
    local=frame.copy();local['p']=np.asarray(pred,float)
    local['err']=(local.y_true-local.p).abs()
    lo=local.ci_low.to_numpy(float);hi=local.ci_high.to_numpy(float);p=local.p.to_numpy(float)
    valid=np.isfinite(lo)&np.isfinite(hi)&(lo<=hi)
    dist=np.maximum(lo-p,0)+np.maximum(p-hi,0)
    local['excess']=np.where(valid,dist,np.nan)
    out={'macro_mae':float(local.groupby('isoform').err.mean().reindex(ISOS).mean()),
         'macro_interval_excess':float(local.groupby('isoform').excess.mean().reindex(ISOS).mean()),
         'n':len(local)}
    if original is not None:
        out['mae_improvement']=float(metric_stats(frame,original)['macro_mae']-out['macro_mae'])
    return out

def safe_auc(y,p):
    return float(roc_auc_score(y,p)) if len(set(np.asarray(y,dtype=int)))==2 else np.nan

def sample_bootstrap(all_preds,baseline_col='base',B=300,seed=2026):
    p=all_preds.copy(); p['benefit']=(p.y_true-p[baseline_col]).abs()-(p.y_true-p.y_pred).abs()
    ss=p.groupby(['scaffold_group','isoform']).benefit.agg(['sum','count']).reset_index()
    sums=ss.pivot(index='scaffold_group',columns='isoform',values='sum').reindex(columns=ISOS).fillna(0).to_numpy()
    cnt=ss.pivot(index='scaffold_group',columns='isoform',values='count').reindex(columns=ISOS).fillna(0).to_numpy()
    rng=np.random.default_rng(seed);v=[]
    for _ in range(B):
        ind=rng.integers(0,len(sums),len(sums));nr=sums[ind].sum(0);dn=cnt[ind].sum(0)
        if (dn>0).all():v.append(float(np.mean(nr/dn)))
    return {'gain':float(p.groupby('isoform').benefit.mean().reindex(ISOS).mean()),
            'low':float(np.quantile(v,.025)) if len(v) else np.nan,
            'high':float(np.quantile(v,.975)) if len(v) else np.nan,'reps':len(v)}

def wrap_results(output,archive_filename,files):
    import zipfile
    with zipfile.ZipFile(archive_filename,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for fn in files:
            path=Path(output)/fn
            if path.exists():z.write(path,arcname='cyp_regime_results/'+fn)
    print('Wrote',archive_filename)

def guarded_folds(train,held):
    assert not set(train.canonical_smiles)&set(held.canonical_smiles)
    assert not set(train.scaffold_group)&set(held.scaffold_group)

def save_csv(df,path):
    # Normalize floating precision across fresh and cached executions.
    df.to_csv(path,index=False,float_format='%.10g')

def get_root():
    return Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')


In [ ]:
OUTER, INNER, MANIFEST, META11 = load_archive(NB11_RESULTS_ZIP)
FEATURES = feature_table(OUTER, INNER, bits=FP_BITS)
print("Outer observations:", len(OUTER), "Inner meta observations:", len(INNER), "Features:", FEATURES["x"].shape)
print("Source scientifically valid:", META11.get("scientifically_valid_full_model_run"))
assert OUTER.groupby("canonical_smiles").fold.nunique().max()==1

## 3. Fold-safe modeling and ablation
**No outer-fold potency labels are used for fitting, selecting a correction, calibrating a gate, or mining training pairs.** The methods cache each completed fold. Expected outputs include a chosen-method audit, per-CYP performance, and low/high-potency diagnostics.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

torch.set_num_threads(min(4,max(1,os.cpu_count() or 1)))
REG_STRENGTH=.03
SOFTPLUS_TAU=.12
CORRECTION_CAP=1.5
EPOCHS=20 if FAST_MODE else 70
PATIENCE=5 if FAST_MODE else 12
META_SPLITS=2  # 2-fold nested calibration for runtime; outer evaluation remains untouched
LOSS_FAMILIES=['point','interval','hybrid']

# Correct interval-aware numerator; denominator uses the held-out cohort's mean-label anchor.
def macro_st_rae(frame,pred):
    a=frame.copy();a['pred']=np.asarray(pred,float)
    values=[]
    for iso,g in a.groupby('isoform'):
        yp=g.pred.to_numpy(float);lo=g.ci_low.to_numpy(float);hi=g.ci_high.to_numpy(float)
        anchor=np.full(len(g),g.y_true.mean())
        numerator=np.maximum(lo-yp,0).sum()+np.maximum(yp-hi,0).sum()
        den=np.maximum(lo-anchor,0).sum()+np.maximum(anchor-hi,0).sum()
        values.append(numerator/den if den>1e-10 else np.nan)
    return float(np.nanmean(values))

class AffineInterval:
    def __init__(self,kind='interval'):
        self.kind=kind;self.param={}
    def fit(self,frame):
        for iso,part in frame.groupby('isoform'):
            b=part.base.to_numpy(float);y=part.y_true.to_numpy(float)
            lo=part.ci_low.to_numpy(float);hi=part.ci_high.to_numpy(float)
            def loss(v):
                yp=v[0]+v[1]*b
                dist=np.maximum(lo-yp,0)+np.maximum(yp-hi,0)
                point=np.sqrt((yp-y)**2+.02**2)-.02
                smooth=np.sqrt(dist**2+.015**2)-.015
                obj=(.75*np.mean(smooth)+.25*np.mean(point)) if self.kind=='interval' else np.mean(point)
                return float(obj+REG_STRENGTH*((v[0])**2+(v[1]-1)**2*5))
            ret=minimize(loss,[0,1.],method='L-BFGS-B',bounds=[(-2,2),(.65,1.35)])
            if not ret.success and not np.isfinite(ret.fun):raise ValueError('Affine calibration optimization failure')
            self.param[iso]=ret.x
        return self
    def predict(self,frame):
        a=np.array([self.param[iso] for iso in frame.isoform]);return a[:,0]+a[:,1]*frame.base.to_numpy(float)

class GatedResidualNet(nn.Module):
    def __init__(self,n_inputs,hidden=24):
        super().__init__()
        self.backbone=nn.Sequential(nn.Linear(n_inputs,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.gate=nn.Linear(hidden,2)
        self.experts=nn.Linear(hidden,2)
    def forward(self,x,b):
        h=self.backbone(x);p=torch.softmax(self.gate(h),dim=1)
        residual=(p*(CORRECTION_CAP*torch.tanh(self.experts(h)))).sum(1)
        return b+residual,p,residual

def torch_data(df,feat,scaler=None):
    raw=make_x(df,feat)
    onehot=np.eye(4,dtype=np.float32)[np.array([ISOS.index(x) for x in df.isoform])]
    raw=np.column_stack([raw,onehot])
    if scaler is None:scaler=StandardScaler().fit(raw)
    x=np.clip(scaler.transform(raw),-8,8).astype(np.float32)
    return x,scaler

def fit_net(frame,features,loss_family='hybrid',seed=2026):
    torch.manual_seed(seed);np.random.seed(seed)
    x,scaler=torch_data(frame,features);ids=np.arange(len(frame))
    # Early stop on scaffold-independent inner-meta dev split.
    if frame.scaffold_group.nunique()>=6:
        tr,va=next(GroupShuffleSplit(test_size=.16,n_splits=1,random_state=seed).split(ids,groups=frame.scaffold_group))
    else:tr=ids;va=ids[:min(5,len(ids))]
    model=GatedResidualNet(x.shape[1]);opt=torch.optim.AdamW(model.parameters(),lr=.002,weight_decay=.015)
    xx=torch.tensor(x);bb=torch.tensor(frame.base.to_numpy(np.float32));yy=torch.tensor(frame.y_true.to_numpy(np.float32))
    ll=torch.tensor(frame.ci_low.to_numpy(np.float32));uu=torch.tensor(frame.ci_high.to_numpy(np.float32))
    thresholds={iso:float(np.quantile(frame.iloc[tr].loc[frame.iloc[tr].isoform==iso,'y_true'],.65))
                for iso in ISOS if (frame.iloc[tr].isoform==iso).sum()>0}
    aux=torch.tensor([float(y>=thresholds.get(iso,5.)) for y,iso in zip(frame.y_true,frame.isoform)],dtype=torch.float32)
    iso_idx=np.array([ISOS.index(v) for v in frame.isoform]); w=np.asarray([len(frame)/(4*max(1,(iso_idx==i).sum())) for i in iso_idx],np.float32);w=torch.tensor(w)
    def component(indices):
        ypred,probs,res=model(xx[indices],bb[indices]);distance=F.softplus((ll[indices]-ypred)/SOFTPLUS_TAU)*SOFTPLUS_TAU+F.softplus((ypred-uu[indices])/SOFTPLUS_TAU)*SOFTPLUS_TAU
        huber=F.smooth_l1_loss(ypred,yy[indices],beta=.25,reduction='none')
        if loss_family=='point':core=huber
        elif loss_family=='interval':core=distance+.1*huber
        else:core=.7*distance+.3*huber
        # Gate supervision is explicit and shared across isoforms; no true gate label at inference.
        gate_ce=F.binary_cross_entropy(probs[:,1].clamp(1e-6,1-1e-6),aux[indices],reduction='none')
        return ((core+.04*gate_ce+.04*res**2)*w[indices]).mean()
    best=float('inf');pat=0;state=None
    for e in range(EPOCHS):
        model.train();opt.zero_grad();loss=component(tr);loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),2.);opt.step()
        model.eval()
        with torch.no_grad():dev=float(component(va))
        if dev<best-1e-5:best=dev;pat=0;state={k:v.detach().clone() for k,v in model.state_dict().items()}
        else:
            pat+=1
            if pat>=PATIENCE:break
    if state is not None:model.load_state_dict(state)
    model.eval();return model,scaler,{'best_dev_loss':best,'epochs_completed':e+1,'train':len(tr),'dev':len(va)}

def net_predict(model,scale,frame,feat):
    raw,_=torch_data(frame,feat,scale)
    with torch.no_grad(): pred,p,res=model(torch.tensor(raw),torch.tensor(frame.base.to_numpy(np.float32)))
    return pred.numpy().astype(float),p.numpy()[:,1].astype(float)

def crossfit_17(train,features,seed):
    splitter=GroupKFold(n_splits=min(META_SPLITS,train.scaffold_group.nunique()))
    prediction={k:np.full(len(train),np.nan) for k in ['AffinePoint','AffineInterval','RegimePoint','RegimeInterval','RegimeHybrid']}
    for n,(t,v) in enumerate(splitter.split(train,groups=train.scaffold_group)):
        a=train.iloc[t];b=train.iloc[v];guarded_folds(a,b)
        for kind,name in [('point','AffinePoint'),('interval','AffineInterval')]:
            prediction[name][v]=AffineInterval(kind).fit(a).predict(b)
        for kind,name in [('point','RegimePoint'),('interval','RegimeInterval'),('hybrid','RegimeHybrid')]:
            model,scale,_=fit_net(a,features,kind,seed+n+100)
            prediction[name][v],_=net_predict(model,scale,b,features)
    scores={k:macro_st_rae(train,v) for k,v in prediction.items()}
    return scores

archive_sha=hashlib.sha256(Path(NB11_RESULTS_ZIP).read_bytes()).hexdigest()[:12]
settings={'tag':EXPERIMENT_TAG,'sha':archive_sha,'fast':FAST_MODE,'fp':FP_BITS,'epoch':EPOCHS,
          'patience':PATIENCE,'smoothing':SOFTPLUS_TAU,'reg':REG_STRENGTH,'meta':META_SPLITS,'seed':SEED}
sig=hashlib.sha256(json.dumps(settings,sort_keys=True).encode()).hexdigest()[:16]
CACHE=OUTPUT/'cache17'/sig;CACHE.mkdir(parents=True,exist_ok=True)
allp=[];allsel=[];allaudit=[]
for fold in FOLDS_TO_RUN:
    cp=CACHE/f'{fold}_pred.csv';sp=CACHE/f'{fold}_select.csv';ap=CACHE/f'{fold}_audit.csv'
    if all(p.exists() for p in [cp,sp,ap]):
        allp.append(pd.read_csv(cp));allsel.append(pd.read_csv(sp));allaudit.append(pd.read_csv(ap));print('Cache hit:',fold);continue
    tr=INNER[INNER.fold==fold].reset_index(drop=True);te=OUTER[OUTER.fold==fold].reset_index(drop=True)
    guarded_folds(tr,te)
    scores=crossfit_17(tr,FEATURES,SEED+fold*7)
    best=min(scores,key=lambda k:(scores[k],k))
    estimated={};diagnostics=[]
    for kind,name in [('point','AffinePoint'),('interval','AffineInterval')]:
        estimated[name]=AffineInterval(kind).fit(tr).predict(te)
    for kind,name in [('point','RegimePoint'),('interval','RegimeInterval'),('hybrid','RegimeHybrid')]:
        model,scale,details=fit_net(tr,FEATURES,kind,SEED+fold*7+555)
        estimated[name],gate=net_predict(model,scale,te,FEATURES)
        diagnostics.append({'fold':fold,'name':name,**details,'mean_gate':float(np.mean(gate))})
    estimated['Baseline']=te.base.to_numpy(float)
    estimated['NestedSelected']=estimated[best]
    rec=[]
    for name,pp in estimated.items():
        for i,row in te.iterrows():rec.append({'fold':fold,'isoform':row.isoform,
            'canonical_smiles':row.canonical_smiles,'scaffold_group':row.scaffold_group,'y_true':row.y_true,
            'ci_low':row.ci_low,'ci_high':row.ci_high,'base':row.base,'y_pred':float(pp[i]),'method':name})
    select=[{'fold':fold,'method':name,'inner_macro_st_rae':score,'selected':name==best} for name,score in scores.items()]
    audit=[{'fold':fold,'fit_molecules':tr.canonical_smiles.nunique(),'held_molecules':te.canonical_smiles.nunique(),
            'scaffold_overlap':len(set(tr.scaffold_group)&set(te.scaffold_group)),'selected':best}]+diagnostics
    for obj,path in [(rec,cp),(select,sp),(audit,ap)]:save_csv(pd.DataFrame(obj),path)
    allp.append(pd.read_csv(cp));allsel.append(pd.read_csv(sp));allaudit.append(pd.read_csv(ap));print('Finished outer fold',fold,'selected',best)

pred=pd.concat(allp,ignore_index=True);select=pd.concat(allsel,ignore_index=True);audit=pd.concat(allaudit,ignore_index=True)
assert not pred.duplicated(['fold','isoform','canonical_smiles','method']).any()
summary=[];per_cyp=[];extreme=[]
for model,df in pred.groupby('method'):
    s=metric_stats(df,df.y_pred);s['macro_st_rae']=macro_st_rae(df,df.y_pred)
    summary.append({'method':model,**s})
    for iso,g in df.groupby('isoform'):
        per_cyp.append({'method':model,'isoform':iso,'n':len(g),'mae':float(np.mean(abs(g.y_true-g.y_pred))),
                        'st_rae':macro_st_rae(g,g.y_pred)})
        q1,q9=np.quantile(g.y_true,[.1,.9])
        for label,part in [('weak',g[g.y_true<=q1]),('potent',g[g.y_true>=q9])]:
            extreme.append({'method':model,'isoform':iso,'region':label,'n':len(part),
               'mae':float(np.mean(abs(part.y_true-part.y_pred))),'bias':float(np.mean(part.y_pred-part.y_true))})
summary=pd.DataFrame(summary).sort_values('macro_st_rae')
for name,df in [('outer_predictions.csv',pred),('macro_summary.csv',summary),('per_isoform.csv',pd.DataFrame(per_cyp)),
                ('activity_extremes.csv',pd.DataFrame(extreme)),('nested_selection.csv',select),('leakage_training_audit.csv',audit)]:save_csv(df,OUTPUT/name)
boot=sample_bootstrap(pred[pred.method=='NestedSelected'],B=150 if FAST_MODE else 1000)
save_csv(pd.DataFrame([boot]),OUTPUT/'paired_mae_bootstrap.csv')
(OUTPUT/'run_metadata.json').write_text(json.dumps({'experiment':'interval-aware + isoform-conditioned gated expert',
 'source':'Notebook11 inner OOF for meta-training; outer OOF for evaluation',
 'folds':FOLDS_TO_RUN,'complete':set(FOLDS_TO_RUN)==set(range(5)),
 'selection':'Group-scaffold meta CV minimizing interval ST-RAE; no outer labels used for selection',
 'warning':'Calibration operates on saved model predictions, not end-to-end CheMeleon fine-tuning. Meta validation first-stage predictions are inner OOF but may have overlapping base-fit histories.',
 'settings':settings},indent=2))
wrap_results(OUTPUT,'cyp_notebook_17_results.zip',['outer_predictions.csv','macro_summary.csv','per_isoform.csv','activity_extremes.csv','nested_selection.csv','leakage_training_audit.csv','paired_mae_bootstrap.csv','run_metadata.json'])
print(summary.to_string(index=False))


## 4. Visual comparison and research interpretation
Inspect the outer-fold tables before drawing conclusions. Improvements are exploratory unless repeated across seeds and split strategies. In particular, compare potent-decile error, interval-aware performance, gate/contrastive diagnostics, and robustness to removing each modeling component.

In [ ]:
import matplotlib.pyplot as plt
try:
    chart_df = summary.sort_values('macro_mae' if 'macro_mae' in summary else 'macro_st_rae')
    key = 'macro_st_rae' if 'macro_st_rae' in summary else 'macro_mae'
    fig,ax=plt.subplots(figsize=(9,4))
    ax.barh(chart_df.method[::-1],chart_df[key][::-1])
    ax.set_xlabel(key+' (lower is better)');ax.set_title('Nested outer-fold comparison')
    plt.tight_layout();plt.show()
except Exception as e:
    print('Optional figure unavailable:',type(e).__name__,str(e))


## 5. Reproducibility checklist
- [ ] Real Notebook 11 archive, no proxy checkpoint
- [ ] `FAST_MODE=False`, outer folds 0–4
- [ ] No held-out scaffold overlaps in audits
- [ ] No selecting thresholds/strengths from outer-fold outcomes
- [ ] Compare against the **fixed** 50:50 CheMeleon + screening baseline on exactly matched molecules
- [ ] Publish a negative result if the specialists fail to improve the strong baseline
- [ ] Record dataset versions, code revisions, and multiple seeds for further research